# 🧠 การเทรนโมเดล AI อ่านและแปลอักษรล้านนา (Lanna Tai Tham OCR Model Training)
### รองรับการรันฟรี 100% บน Google Colab (T4 GPU Free Tier)
---

In [ ]:
# 1. ติดตั้ง Library ที่จำเป็น
!pip install -q torch torchvision torchaudio transformers datasets pillow pandas tqdm evaluate

In [ ]:
# 2. ตรวจสอบ GPU
import torch
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    print('⚠️ กำลังรันบน CPU (แนะนำให้ไปที่ เมนู Runtime -> Change runtime type -> เลือก T4 GPU เพื่อความรวดเร็ว)')

In [ ]:
# 3. แตกไฟล์ Dataset (อัปโหลดไฟล์ dataset.zip มาไว้ที่โฟลเดอร์ Colab)
!unzip -q dataset.zip -d ./lanna_dataset
print('✅ แตกไฟล์ Dataset เรียบร้อยแล้ว!')

In [ ]:
# 4. โหลดข้อมูล Labels & รูปภาพ
import pandas as pd
import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

df = pd.read_csv('./lanna_dataset/labels.csv')
print('จำนวนข้อมูลทั้งหมด:', len(df))
display(df.head())

In [ ]:
# 5. โหลด TrOCR Processor และ Pre-trained Model (Microsoft TrOCR)
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print('⏳ กำลังโหลด Base TrOCR Model...')
processor = TrOCRProcessor.from_pretrained('microsoft/trocr-base-stage1')
model = VisionEncoderDecoderModel.from_pretrained('microsoft/trocr-base-stage1')

# ปรับแต่ง Tokenizer / Generation Config
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
print('✅ โหลดโมเดลสำเร็จ พร้อมเทรนบน:', device)

In [ ]:
# 6. สร้าง PyTorch Dataset Class
class LannaOcrDataset(Dataset):
    def __init__(self, df, root_dir, processor, max_target_length=64):
        self.df = df
        self.root_dir = root_dir
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.root_dir, row['image_path'])
        image = Image.open(img_path).convert('RGB')
        pixel_values = self.processor(image, return_tensors='pt').pixel_values

        # Label เป็นคำแปลภาษาไทย + อักษรล้านนา
        label_text = f"{row['thai_translation']}"
        labels = self.processor.tokenizer(
            label_text,
            padding='max_length',
            max_length=self.max_target_length,
            truncation=True,
            return_tensors='pt'
        ).input_ids

        labels = [label if label != self.processor.tokenizer.pad_token_id else -100 for label in labels.squeeze()]
        return {
            'pixel_values': pixel_values.squeeze(),
            'labels': torch.tensor(labels)
        }

train_df, val_df = train_test_split(df, test_size=0.15, random_state=42)
train_dataset = LannaOcrDataset(train_df, './lanna_dataset', processor)
val_dataset = LannaOcrDataset(val_df, './lanna_dataset', processor)
print(f'Train samples: {len(train_dataset)}, Validation samples: {len(val_dataset)}')

In [ ]:
# 7. เริ่มการเทรน (Training with HuggingFace Seq2SeqTrainer)
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, default_data_collator

training_args = Seq2SeqTrainingArguments(
    predict_with_generate=True,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    fp16=torch.cuda.is_available(),
    output_dir='./lanna_trocr_model',
    logging_steps=50,
    save_total_limit=2,
    num_train_epochs=5,
    learning_rate=5e-5,
    weight_decay=0.01,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.image_processor,
    args=training_args,
    compute_metrics=None,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=default_data_collator,
)

print('🚀 เริ่มต้นการเทรนโมเดล AI ล้านนา...')
trainer.train()
print('🎉 เทรนเสร็จสมบูรณ์ 100%!')

In [ ]:
# 8. เซฟโมเดล และทดสอบการอ่านรูปภาพจริง
model.save_pretrained('./lanna_ocr_final_model')
processor.save_pretrained('./lanna_ocr_final_model')

# ฟังก์ชันทดสอบทำนายภาพ
def predict_lanna(image_path):
    img = Image.open(image_path).convert('RGB')
    pixels = processor(img, return_tensors='pt').pixel_values.to(device)
    out_ids = model.generate(pixels)
    res_text = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
    return res_text

test_sample = val_df.iloc[0]
pred = predict_lanna(os.path.join('./lanna_dataset', test_sample['image_path']))
print(f"ภาพ: {test_sample['image_path']}")
print(f"เฉลย: {test_sample['thai_translation']}")
print(f"AI ทายว่า: {pred}")

In [ ]:
# 9. บีบอัดไฟล์โมเดลพร้อมดาวน์โหลด
!zip -r lanna_custom_ocr_model.zip ./lanna_ocr_final_model
from google.colab import files
files.download('lanna_custom_ocr_model.zip')
print('📦 โมเดลถูกบีบอัดและพร้อมดาวน์โหลดนำไปใช้งานในแอพ Flutter แล้ว!')